# 09 · Lendo Múltiplos Formatos de Arquivo

**Teoria**: docs/05-pyspark-na-pratica.md

**Pré-requisito**: `make spark` em execução (mesmo cluster Standalone + Spark Connect dos notebooks 06-08).

🎯 **Objetivo**: ler dados de fontes que exportam em formatos diferentes — CSV e JSON/JSON Lines — reconciliar os schemas num único DataFrame, e fechar o ciclo escrevendo o resultado limpo na camada Silver.

**Cenário de negócio**: a empresa coleta avaliações de clientes (`avaliacoes`) por 3 canais, cada um com seu próprio sistema de exportação:

- **App mobile** → exporta em **JSON Lines**, com metadados do dispositivo aninhados
- **Site institucional** → exporta em **CSV** limpo, UTF-8
- **Call Center** (sistema legado) → exporta em **CSV**, mas com separador `;`, encoding Latin-1, datas `dd/mm/aaaa` e números decimais com vírgula — o "Excel brasileiro" clássico

Você vai ler as 3, entender por que cada uma pede opções diferentes, unir tudo num schema comum, responder perguntas de negócio, e gravar o resultado — desta vez em Parquet, na camada Silver.

---
### 🔤 O que você vai praticar

1. **Leitura de JSON / JSON Lines** — `spark.read.json(...)`, incluindo campos aninhados (`structs`)
2. **Leitura de CSV** — opções de `header`, `sep`, `encoding`, `inferSchema`
3. **Diagnóstico de leitura malfeita** — o que acontece quando você usa as opções erradas, e como identificar o problema
4. **Reconciliação de schemas heterogêneos** — `unionByName(allowMissingColumns=True)`
5. **`select`/`filter`/`groupBy`/`agg`** sobre o resultado unificado — as mesmas ferramentas dos notebooks 02-06, agora sobre dados que vieram de fontes bem diferentes
6. **Escrita na camada Silver** — fechando o ciclo Bronze → Silver com Parquet particionado

Vamos começar conectando ao cluster.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

## Lendo `empresas` (Parquet, já conhecido)

Vamos precisar de `empresas` para os `join`s das perguntas de negócio mais adiante — mesma leitura Parquet dos notebooks anteriores, sem novidade aqui.

In [ ]:
sdf_empresas = spark.read.parquet("/data/bronze/empresas")
sdf_empresas.show(5)

## Fonte 1: App mobile (JSON Lines)

Cada avaliação do app vem como um objeto JSON por linha (**JSON Lines**, também chamado NDJSON) — o formato padrão para logs e eventos, porque é fácil de anexar linha por linha sem reescrever o arquivo inteiro.

📌 O app também manda metadados do dispositivo (sistema operacional, versão do app) como **campo aninhado** — algo que CSV simplesmente não consegue representar sem achatar em colunas separadas. É por isso que sistemas de eventos quase sempre usam JSON: o formato aceita estrutura aninhada nativamente.

In [ ]:
sdf_app = spark.read.json("/data/bronze/avaliacoes_app")
sdf_app.printSchema()
sdf_app.show(5, truncate=False)

📌 **Entendendo o schema**: repare que `dispositivo` não é uma coluna de texto — é um `struct` com dois campos (`os`, `versao_app`), lidos diretamente do objeto JSON aninhado. Para acessar um campo de dentro do struct, use notação de ponto: `col("dispositivo.os")`.

💡 **Achatando o struct**: se preferir colunas simples em vez de um struct, extraia com `select`/`withColumn` — é isso que vamos fazer na hora de unificar as 3 fontes.

In [ ]:
from pyspark.sql.functions import col

sdf_app.select(
    "id_avaliacao", "nota", col("dispositivo.os").alias("dispositivo_os"),
    col("dispositivo.versao_app").alias("versao_app"),
).show(5)

## Fonte 2: Site institucional (CSV limpo)

O backend do site exporta um CSV direto de uma consulta SQL: header, vírgula como separador, UTF-8, datas em ISO (`aaaa-mm-dd`) — o "caminho feliz" do CSV.

In [ ]:
sdf_site = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/data/bronze/avaliacoes_site")
)
sdf_site.printSchema()
sdf_site.show(5)

📌 **`inferSchema` tem um custo escondido**: diferente do Parquet (que traz o schema embutido nos metadados do arquivo, sem precisar ler os dados), CSV e JSON são formatos **sem schema** — o Spark não sabe os tipos das colunas até examiná-las. Com `inferSchema=True`, o Spark faz uma **passada extra completa** só para descobrir os tipos, e só depois lê os dados de verdade. Em produção, com arquivos grandes, é comum declarar o schema explicitamente (`StructType`) para evitar pagar essa segunda leitura.

⚠️ **Atenção**: sem `inferSchema`, todas as colunas viram `string` por padrão — `nota` apareceria como texto, não como número, e não daria para comparar ou agregar sem um `cast()` manual antes.

## Fonte 3: Call Center (CSV legado) — quando a leitura "óbvia" quebra

O sistema do call center é o mais antigo dos três — exporta um CSV no estilo clássico de planilha brasileira: separador `;`, encoding Latin-1 (ISO-8859-1, comum em sistemas legados que nunca migraram para UTF-8), datas no formato `dd/mm/aaaa`, e números decimais com vírgula em `tempo_atendimento_min`.

Vamos tentar ler do jeito "óbvio" primeiro — as mesmas opções que funcionaram para o CSV do site — e ver o que acontece.

In [ ]:
# Leitura ingênua — mesmas opções que funcionaram para o CSV do site
sdf_callcenter_quebrado = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/data/bronze/avaliacoes_callcenter")
)
sdf_callcenter_quebrado.printSchema()
sdf_callcenter_quebrado.show(5, truncate=False)

📌 **O que quebrou**: sem `sep=";"`, o Spark tentou separar as colunas por vírgula — mas o *header* não tem nenhuma vírgula, então nada é separado: os 7 nomes de campo (ainda grudados por `;`) viram o nome de uma única coluna gigante, como você vê no `printSchema()`/`show()` acima. O problema aparece nas linhas de dado, que *têm* vírgula: `tempo_atendimento_min` usa vírgula como separador **decimal** (`8,5`), não como delimitador de campo. Como o parser só espera 1 coluna, ele trata essa vírgula como fim do campo e descarta silenciosamente o que vem depois dela — repare que `8,5` virou `8`, `4,0` virou `4` e `28,5` virou `28` no resultado acima.

Repare, aliás, que esse é o próprio motivo de sistemas brasileiros usarem `;` como delimitador em vez de `,`: como a vírgula já é o separador decimal no Brasil, usá-la também para separar colunas criaria exatamente essa ambiguidade.

E mesmo corrigindo o separador, os caracteres acentuados (`ã`, `ç`, `é`) ainda viriam corrompidos sem o `encoding` certo — o arquivo foi escrito em Latin-1, não UTF-8 (o padrão que o Spark assume).

⚠️ **O pior**: essa leitura não lança nenhum erro — ela "funciona" e devolve dado errado silenciosamente. É o tipo de bug que só aparece quando alguém repara que os números não batem.

Vamos corrigir, uma opção de cada vez.

In [ ]:
from pyspark.sql.functions import regexp_replace, to_date

sdf_callcenter = (
    spark.read
    .option("header", True)
    .option("sep", ";")               # separador correto do export legado
    .option("encoding", "ISO-8859-1") # Latin-1 — sem isso, acentos vêm corrompidos
    .option("inferSchema", True)
    .csv("/data/bronze/avaliacoes_callcenter")
)
sdf_callcenter.printSchema()
sdf_callcenter.show(5, truncate=False)

📌 **Ainda faltam dois ajustes**: mesmo com `sep`/`encoding` certos, `data` ainda chega como texto no formato `dd/mm/aaaa` (o Spark só reconhece ISO como data automaticamente) e `tempo_atendimento_min` chega como `string`, porque tem vírgula como separador decimal em vez de ponto.

In [ ]:
sdf_callcenter = (
    sdf_callcenter
    .withColumn("data", to_date(col("data"), "dd/MM/yyyy"))
    .withColumn(
        "tempo_atendimento_min",
        regexp_replace(col("tempo_atendimento_min"), ",", ".").cast("double"),
    )
)
sdf_callcenter.printSchema()
sdf_callcenter.select("id_avaliacao", "data", "tempo_atendimento_min").show(5)

✅ **Agora sim**: `data` é `date` de verdade (dá pra comparar, filtrar por intervalo, extrair ano/mês) e `tempo_atendimento_min` é `double` (dá pra calcular médias). Três fontes, três leituras diferentes — é hora de unificar.

## Reconciliando os 3 schemas

As 3 fontes compartilham um schema canônico — `id_avaliacao, id_empresa, canal, nota, comentario, data` — mas cada uma também carrega uma coluna extra que só ela tem (`versao_app` no app, `tempo_atendimento_min` no call center). Vamos:

1. Selecionar/renomear cada fonte para o schema canônico + suas colunas extras
2. Unir tudo com `unionByName(allowMissingColumns=True)` — que preenche com `NULL` a coluna extra nas fontes onde ela não existe, em vez de dar erro por schemas diferentes

In [ ]:
avaliacoes_app_norm = sdf_app.select(
    "id_avaliacao", "id_empresa", "canal", "nota", "comentario",
    to_date(col("data")).alias("data"),
    col("dispositivo.versao_app").alias("versao_app"),
)
avaliacoes_site_norm = sdf_site.select(
    "id_avaliacao", "id_empresa", "canal", "nota", "comentario", "data",
)
avaliacoes_callcenter_norm = sdf_callcenter.select(
    "id_avaliacao", "id_empresa", "canal", "nota", "comentario", "data",
    "tempo_atendimento_min",
)

avaliacoes = (
    avaliacoes_app_norm
    .unionByName(avaliacoes_site_norm, allowMissingColumns=True)
    .unionByName(avaliacoes_callcenter_norm, allowMissingColumns=True)
)
avaliacoes.printSchema()
print(f"App: {avaliacoes_app_norm.count():,} + Site: {avaliacoes_site_norm.count():,} + "
      f"Call Center: {avaliacoes_callcenter_norm.count():,} = {avaliacoes.count():,} no total")

📌 **Verificação**: a soma das 3 contagens individuais bate exatamente com a contagem do DataFrame unificado — nenhuma linha foi perdida ou duplicada na união. Repare também no schema final: `versao_app` e `tempo_atendimento_min` aparecem como colunas normais, com `NULL` nas linhas que vieram de uma fonte que não tem aquele campo.

## Perguntas de negócio

Com as 3 fontes unificadas, o resto é o que você já pratica desde o notebook 02: `select`, `filter`, `groupBy`, `agg`, `join`.

#### 💡 **Exemplo 1:** Nota média por canal — qual sistema traz os clientes mais satisfeitos?

In [ ]:
from pyspark.sql.functions import avg, count, round as spark_round

avaliacoes.groupBy("canal").agg(
    spark_round(avg("nota"), 2).alias("nota_media"),
    count("*").alias("total_avaliacoes"),
).orderBy(col("nota_media").desc()).show()

📌 **Cuidado com essa leitura**: uma nota média mais baixa num canal não significa necessariamente que o atendimento seja pior nele — pode ser só o canal que atrai clientes já insatisfeitos com outra parte do processo (ex.: quem liga pro Call Center normalmente já está com um problema). Nota média por canal é um ponto de partida, não uma conclusão.

#### 💡 **Exemplo 2:** Nota média por setor — join com `empresas`

In [ ]:
avaliacoes_join_empresas = avaliacoes.join(sdf_empresas, "id_empresa")

avaliacoes_join_empresas.groupBy("setor").agg(
    spark_round(avg("nota"), 2).alias("nota_media"),
    count("*").alias("total_avaliacoes"),
).orderBy(col("nota_media").desc()).show(15)

#### 💡 **Exemplo 3:** Empresas "em alerta" — nota média baixa E volume relevante de avaliações

Uma empresa com nota média 1,5 mas só 2 avaliações não é um alerta confiável — pode ser azar de amostra pequena. O padrão `HAVING` (agregue, depois filtre o resultado — visto no notebook 03) resolve isso: só considerar empresas com volume mínimo de avaliações.

In [ ]:
empresas_em_alerta = (
    avaliacoes_join_empresas.groupBy("id_empresa", "nome_empresa", "setor")
    .agg(
        spark_round(avg("nota"), 2).alias("nota_media"),
        count("*").alias("total_avaliacoes"),
    )
    .filter((col("nota_media") < 3) & (col("total_avaliacoes") >= 20))
    .orderBy("nota_media")
)
empresas_em_alerta.show(10, truncate=False)

#### 💡 **Exemplo 4:** Evolução mensal da nota média — a satisfação está melhorando ou piorando?

In [ ]:
from pyspark.sql.functions import month, year

avaliacoes_por_mes = (
    avaliacoes
    .withColumn("ano", year("data"))
    .withColumn("mes", month("data"))
    .groupBy("ano", "mes")
    .agg(spark_round(avg("nota"), 2).alias("nota_media"), count("*").alias("total"))
    .orderBy("ano", "mes")
)
avaliacoes_por_mes.show(24)

#### 💡 **Exemplo 5 (bônus):** Existe uma versão do app com mais reclamações?

Essa pergunta só dá pra responder porque preservamos `versao_app` na união — se tivéssemos descartado as colunas exclusivas de cada fonte, essa informação teria se perdido.

In [ ]:
avaliacoes.filter(col("versao_app").isNotNull()).groupBy("versao_app").agg(
    spark_round(avg("nota"), 2).alias("nota_media"),
    count("*").alias("total_avaliacoes"),
).orderBy("nota_media").show()

📌 **Achado**: uma versão específica do app puxa a nota média bem abaixo das demais — um padrão clássico de "essa versão tem um bug" que só aparece quando você segmenta por versão em vez de olhar a média geral do canal "App".

#### 💡 **Exemplo 6 (bônus):** Atendimentos mais demorados recebem notas piores?

In [ ]:
avaliacoes.filter(col("tempo_atendimento_min").isNotNull()).groupBy("nota").agg(
    spark_round(avg("tempo_atendimento_min"), 1).alias("tempo_medio_min"),
).orderBy("nota").show()

📌 **Achado**: o tempo médio de atendimento cai conforme a nota sobe — atendimentos mais rápidos tendem a gerar avaliações melhores no Call Center. Mais um motivo de negócio para investir em reduzir o tempo de espera.

## Fechando o ciclo: Bronze → Silver

Até aqui, `avaliacoes` só existe como DataFrame em memória — se o kernel reiniciar, tudo isso precisa rodar de novo. Agora que os 3 formatos foram lidos, corrigidos e unificados num schema único e tipado, esse é o momento certo para persistir o resultado como **Parquet** na camada Silver: colunar, com schema embutido, sem mais ambiguidade de delimitador, encoding ou formato de data.

📌 Este é o primeiro notebook do lab que escreve de verdade na camada Silver — os notebooks anteriores só leram Bronze.

In [ ]:
from pyspark.sql.functions import year as spark_year

(
    avaliacoes
    .withColumn("ano", spark_year("data"))
    .write.mode("overwrite")
    .partitionBy("ano")
    .parquet("/data/silver/avaliacoes")
)
print("✅ Silver layer written to /data/silver/avaliacoes (partitioned by ano)")

📌 **Verificação rápida**: ler de volta o que acabou de ser escrito, confirmando que a contagem bate com o DataFrame unificado em memória.

In [ ]:
sdf_silver = spark.read.parquet("/data/silver/avaliacoes")
print(f"Linhas na Silver: {sdf_silver.count():,} (esperado: {avaliacoes.count():,})")
sdf_silver.show(5)

---
🎉 **Parabéns!** Você completou o notebook 09.

Você aprendeu:
- Ler **JSON Lines** com campos aninhados (`struct`) e "achatar" com notação de ponto
- Ler **CSV** com opções de `header`, `sep`, `encoding` e `inferSchema` — e por que `inferSchema` custa uma leitura extra
- Diagnosticar uma leitura de CSV malfeita (separador errado, encoding errado, data em formato não-ISO, decimal com vírgula) e corrigi-la opção por opção
- Reconciliar schemas heterogêneos com `unionByName(allowMissingColumns=True)`, preservando colunas exclusivas de cada fonte
- Responder perguntas de negócio (`select`/`filter`/`groupBy`/`agg`/`join`) sobre dados vindos de fontes e formatos diferentes
- Escrever a primeira camada **Silver** do lab — Parquet particionado, resultado de reconciliar 3 formatos bagunçados num schema único